# Survival Analysis After Endodontic Treatment (Compact Statistical Notebook)

This notebook produces a **compact, reproducible** analysis for endodontic outcomes, including:

- Data import & cleaning (single, robust pipeline)
- Episode construction (one episode per *patient × tooth × treatment type*)
- Descriptive outcomes (**success / failure**)
- Population description (Table 1–style)
- **Kaplan–Meier** survival curves
- **Log-rank** tests (overall + pairwise with multiplicity correction)
- **Chi-square** tests (categorical comparisons)
- **Cox proportional hazards** models (overall + per cohort)

All table/figure titles are **English** and proofed for spelling/terminology.


## ⚠️ תיקון קריטי — STUDY_END

**הבעיה שזוהתה:** המחברת המקורית השתמשה ב-`STUDY_END = 2025-05-05`, 
אך שליפת ה-SQL מגבילה את כל הנתונים ל-`DueDate <= '2021-01-01'`. 
לא קיימים נתונים מ-2022–2024.

**התיקון:** `STUDY_END` שונה ל-`2021-01-01` (תא Parameters למטה).

**השלכות על המאמר לאחר הרצה מחדש:**
- כל מספרי ה-follow-up ישתנו (median, mean±SD)
- עקומות KM ומודל Cox — ייתכן שישתנו
- יש לתקן במאמר: '2014–2024' → '2014–2020', 'ten-year period' → 'seven-year period'
- תאריך נעילת המסד במאמר: לשנות ל-1 January 2021


In [ ]:
# If needed (run once):
# !pip install lifelines openpyxl scipy


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test


In [ ]:
# -----------------------
# Parameters (edit as needed)
# -----------------------

# Option A: set a local path
DATA_PATH = r"C:\Users\cahana_s\research\research_RCT_maria\data_old\RCT_data.xlsx"


STUDY_END = pd.Timestamp("2021-01-01")  # censoring date — data locked at 2021-01-01 (SQL query: DueDate <= '2021-01-01')
WINDOW_DAYS = 365                       # post-start window for coronal restoration indicators


In [ ]:
print(" window:", WINDOW_DAYS, " days")

In [ ]:
# -----------------------
# Load
# -----------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "DATA_PATH not found. Please edit DATA_PATH to point to the Excel file on your machine."
    )

raw = pd.read_excel(DATA_PATH)

print("Raw shape:", raw.shape)
print("Raw columns (first 30):", list(raw.columns)[:30])
cols_to_hide = ["Patient_ID"]  # replace with actual column names
display(raw.drop(columns=cols_to_hide).head())


In [ ]:
# -----------------------
# Cleaning helpers
# -----------------------
def to_dt(s):
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def safe_str(s):
    return s.fillna("").astype(str)

def is_Y(series):
    return (safe_str(series).str.strip().str.upper() == "Y").astype(int)

def contains(series, pattern):
    return safe_str(series).str.contains(pattern, regex=True, na=False)

# Date parsing (guarded)
raw["Treatment_Date_dt"] = to_dt(raw.get("Treatment_Date"))
raw["Failure_Treatment_Date_dt"] = to_dt(raw.get("Failure_Treatment_Date"))
raw["First_Initial_Treatment_Date_dt"] = to_dt(raw.get("First_Initial_Treatment_Date"))

# Core IDs
raw["Patient_ID"] = pd.to_numeric(raw.get("Patient_ID"), errors="coerce").astype("Int64")
raw["Tooth_Num"] = pd.to_numeric(raw.get("Tooth_Num"), errors="coerce").astype("Int64")

# Outcome (row-level) normalization if present
raw["Treatment_Status_norm"] = safe_str(raw.get("Treatment_Status")).str.strip().str.lower()
raw["failure_flag_row"] = (
    raw["Failure_Treatment_Date_dt"].notna()
    | raw.get("failure_code").notna()
    | (raw["Treatment_Status_norm"] == "failure")
).astype(int)

print("Row-level failures:", int(raw["failure_flag_row"].sum()))


In [ ]:
# -----------------------
# Cohort assignment (treatment type)
# -----------------------
# Uses Hebrew text in 'סיווג' (if present) to map into English cohort labels.

# if "סיווג" not in raw.columns:
#     raise KeyError(
#         "Column 'סיווג' was not found in the current source file. "
#         f"Available columns: {list(raw.columns)}. "
#         "A cohort-classification source is required before the analytic episodes and PROBE audit can be rebuilt."
#     )

raw["סיווג"] = safe_str(raw["סיווג"] )

def assign_cohort(text):
    t = str(text)
    tl = t.lower()
    if "אפיס" in t or "אפיסקט" in t or "apic" in tl:
        return "Apicoectomy"
    if "חידוש" in t:
        return "Root canal retreatment"
    if "טיפול שורש" in t:
        return "Root canal treatment"
    return None

raw["Cohort"] = raw["סיווג"].apply(assign_cohort)

print("Rows per cohort (raw):")
display(raw["Cohort"].value_counts(dropna=False))


In [ ]:
# -----------------------
# Row-level coronal restoration indicators (from 'סיווג')
# -----------------------
# Sealing / build-up patterns, crown patterns.


sealing_pat = r"איטום|מבנה"
crown_pat   = r"הכתר|כתר"

raw["Has_Sealing_row"] = contains(raw["סיווג"], sealing_pat).astype(int)
raw["Has_Crown_row"]   = contains(raw["סיווג"], crown_pat).astype(int)

print("Row-level sealing prevalence:", float(raw["Has_Sealing_row"].mean()))
print("Row-level crown prevalence:", float(raw["Has_Crown_row"].mean()))


In [ ]:
# -----------------------
# Patient-level covariates (Table 1 / Cox)
# -----------------------
patient_cols = ["Patient_ID", "Mac_Gender", "Age_In_Treatment"]
keep_cols = [c for c in patient_cols if c in raw.columns]

patient_base = (raw.dropna(subset=["Patient_ID"])
                  .sort_values("Treatment_Date_dt")
                  .groupby("Patient_ID", as_index=False)
                  .agg({c: "first" for c in keep_cols}))

# Sex coding: ז=male, נ=female
patient_base["Male"] = (safe_str(patient_base.get("Mac_Gender")).str.strip() == "ז").astype(int)

# Age group with 40–60 reference
patient_base["Age"] = pd.to_numeric(patient_base.get("Age_In_Treatment"), errors="coerce")
patient_base["AgeGroup"] = pd.cut(
    patient_base["Age"],
    bins=[-np.inf, 40, 60, np.inf],
    labels=["<40", "40–60", "≥60"]
)

# Systemic flags (guarded)
sys_map = {
    "Smoking": "Smok_No",
    "Cancer": "Cancer_No",
    "Diabetes": "Diabet_No",
    "Biphos_use": "Biphos",
    "Pregnancy": "Pregnancy_No",
    "Allergy": "Allergy_Yes",
    "HeartDisease": "Heart_no",
}
present_sys = {k:v for k,v in sys_map.items() if v in raw.columns}

patient_sys = raw.dropna(subset=["Patient_ID"])[["Patient_ID"] + list(present_sys.values())].copy()

for out, col in present_sys.items():
    patient_sys[out] = is_Y(raw[col])

# Hypertension: Hipertonia_Yes = Y means NO hypertension
if "Hipertonia_Yes" in raw.columns:
    no_htn = is_Y(raw["Hipertonia_Yes"])
    patient_sys["Hypertension"] = (1 - no_htn).where(raw["Hipertonia_Yes"].notna(), 0).astype(int)
    present_sys["Hypertension"] = "Hipertonia_Yes"  # marker only

agg_cols = [c for c in list(present_sys.keys()) if c in patient_sys.columns]
patient_sys = patient_sys[["Patient_ID"] + agg_cols].groupby("Patient_ID", as_index=False).max()

patient = patient_base.merge(patient_sys, on="Patient_ID", how="left")

systemic_cols = [c for c in ["Smoking","Cancer","Diabetes","Biphos_use","Pregnancy","Allergy","HeartDisease","Hypertension"] if c in patient.columns]

print("Patient table shape:", patient.shape)
cols_to_hide = ["Patient_ID"]  # replace with actual column names
display(patient.drop(columns=cols_to_hide).head())


In [ ]:
# -----------------------
# Build episodes (ONE per Patient_ID × Tooth_Num × Cohort)
# -----------------------
# Start: first Treatment_Date_dt in cohort
# Event: earliest Failure_Treatment_Date_dt >= start (if any)
# Stop: failure date if event else STUDY_END
# Duration: stop - start (days)

cohort_rows = raw.dropna(subset=["Patient_ID","Tooth_Num","Treatment_Date_dt","Cohort"]).copy()

starts = (cohort_rows.groupby(["Patient_ID","Tooth_Num","Cohort"], as_index=False)
          .agg(start_date=("Treatment_Date_dt","min")))

tmp = cohort_rows.merge(starts, on=["Patient_ID","Tooth_Num","Cohort"], how="inner")
tmp = tmp[tmp["Failure_Treatment_Date_dt"].notna()].copy()
tmp = tmp[tmp["Failure_Treatment_Date_dt"] >= tmp["start_date"]].copy()

min_fail = (tmp.groupby(["Patient_ID","Tooth_Num","Cohort"], as_index=False)
            .agg(failure_date=("Failure_Treatment_Date_dt","min")))

episodes = starts.merge(min_fail, on=["Patient_ID","Tooth_Num","Cohort"], how="left")
episodes["event"] = episodes["failure_date"].notna().astype(int)
episodes["stop_date"] = pd.to_datetime(episodes["failure_date"].fillna(STUDY_END), errors="coerce")
episodes["duration_days"] = (episodes["stop_date"] - pd.to_datetime(episodes["start_date"], errors="coerce")).dt.days

episodes["episode_id"] = (
    episodes["Patient_ID"].astype(str) + "|" +
    episodes["Tooth_Num"].astype(str) + "|" +
    episodes["Cohort"].astype(str) + "|" +
    pd.to_datetime(episodes["start_date"]).dt.strftime("%Y-%m-%d")
)

episodes = episodes.dropna(subset=["duration_days"]).copy()
episodes = episodes[episodes["duration_days"] >= 0].copy()

print("Episodes shape:", episodes.shape)
display(episodes.groupby("Cohort")["event"].agg(Episodes="count", Failures="sum", FailureRate="mean"))


In [ ]:
# -----------------------
# Episode-level coronal restoration flags within window after start
# -----------------------
episodes = episodes.copy()
episodes["start_date"] = pd.to_datetime(episodes["start_date"], errors="coerce")
episodes["stop_date"]  = pd.to_datetime(episodes["stop_date"], errors="coerce")

episodes["_window_end"] = (episodes["start_date"] + pd.Timedelta(days=WINDOW_DAYS))
episodes["_window_end"] = episodes[["_window_end","stop_date"]].min(axis=1)

tmp = episodes[["episode_id","Patient_ID","Tooth_Num","start_date","_window_end"]].merge(
    raw[["Patient_ID","Tooth_Num","Treatment_Date_dt","Has_Sealing_row","Has_Crown_row"]],
    on=["Patient_ID","Tooth_Num"],
    how="left"
)

in_window = (
    tmp["Treatment_Date_dt"].notna() & tmp["start_date"].notna() & tmp["_window_end"].notna() &
    (tmp["Treatment_Date_dt"] >= tmp["start_date"]) & (tmp["Treatment_Date_dt"] <= tmp["_window_end"])
)

tmpw = tmp.loc[in_window].copy()

resto_ep = (tmpw.groupby("episode_id", as_index=False)
            .agg(
                Has_Sealing_in_window=("Has_Sealing_row","max"),
                Has_Crown_in_window=("Has_Crown_row","max"),
            ))

episodes = episodes.drop(columns=["Has_Sealing_in_window","Has_Crown_in_window"], errors="ignore")
episodes = episodes.merge(resto_ep, on="episode_id", how="left")
episodes["Has_Sealing_in_window"] = episodes["Has_Sealing_in_window"].fillna(0).astype(int)
episodes["Has_Crown_in_window"]   = episodes["Has_Crown_in_window"].fillna(0).astype(int)

# Group label (used for plots/tests/models)
episodes["Coronal_Restoration_Group"] = "Neither"
episodes.loc[(episodes["Has_Sealing_in_window"]==1) & (episodes["Has_Crown_in_window"]==0), "Coronal_Restoration_Group"] = "Sealing only"
episodes.loc[(episodes["Has_Crown_in_window"]==1), "Coronal_Restoration_Group"] = "Sealing + Crown"

episodes = episodes.drop(columns=["_window_end"], errors="ignore")

print("Coronal restoration groups (episode-level):")
display(episodes["Coronal_Restoration_Group"].value_counts(dropna=False))


In [ ]:
# -----------------------
# Final analytic dataset: merge patient covariates into episodes
# -----------------------
episodes = episodes.merge(patient.drop(columns=[c for c in ["Mac_Gender","Age_In_Treatment"] if c in patient.columns]),
                          on="Patient_ID", how="left")

print("Analytic dataset shape:", episodes.shape)
cols_to_hide = ["Patient_ID"]  # replace with actual column names
display(episodes.drop(columns=cols_to_hide).head())

In [ ]:
# -----------------------
# 0) Variable inventory (single source of truth)
# -----------------------
# Goal: use the SAME variable lists across:
# - Descriptives (success/failure rates)
# - Chi-square/Fisher (proportions; ignores time)
# - Kaplan–Meier plotting helpers
# - Log-rank (time-to-event; unadjusted)
# - Cox (time-to-event; adjusted)

# If you already have the exact Cox covariate names (e.g., from your modeling pipeline),
# put them here so EVERY step is consistent.
# Example: COX_COVARIATES = [...]
COX_COVARIATES = globals().get("cox_covariates", None)

# Columns used by survival/outcome
ID_COL = "Patient_ID"
TIME_COL = "duration_days"
EVENT_COL = "event"
COHORT_COL = "Cohort"

def _binary_01(s):
    s2 = pd.to_numeric(s, errors="coerce")
    u = set(s2.dropna().unique().tolist())
    return len(u) >= 1 and u.issubset({0,1})

def build_variable_lists(df, cox_covariates=None, exclude=None):
    if exclude is None:
        exclude = set()
    exclude = set(exclude) | {ID_COL, TIME_COL, EVENT_COL}

    # 1) Cox covariates (as given) OR fallback to all 0/1 dummies + key categoricals
    if cox_covariates is not None:
        cox_list = [c for c in cox_covariates if c in df.columns and c not in exclude]
    else:
        cox_list = []
        for c in df.columns:
            if c in exclude:
                continue
            if _binary_01(df[c]):
                cox_list.append(c)

    # 2) Categorical vars for Chi2/Log-rank:
    # Prefer "pre-dummified" categoricals if present; otherwise use 0/1 dummies.
    categorical = []
    for c in df.columns:
        if c in exclude:
            continue
        if c == COHORT_COL:
            categorical.append(c)
            continue
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]):
            categorical.append(c)

    # If none found (or you mostly work with dummies), fall back to 0/1 dummies list
    if len(categorical) == 0:
        categorical = [c for c in cox_list if _binary_01(df[c])]

    # 3) Binary/systemic list (for Table-1 style %)
    binary = [c for c in df.columns if c not in exclude and _binary_01(df[c])]

    return {
        "cox_covariates": cox_list,
        "categorical_vars": categorical,
        "binary_vars": binary,
    }

VARLISTS = build_variable_lists(episodes, cox_covariates=COX_COVARIATES, exclude={COHORT_COL})
COX_VARS = VARLISTS["cox_covariates"]
CAT_VARS = VARLISTS["categorical_vars"]
BIN_VARS = VARLISTS["binary_vars"]

print("Variable inventory:")
print(f"- Cox covariates: {len(COX_VARS)}")
print(f"- Categorical vars (for chi2/log-rank): {len(CAT_VARS)}")
print(f"- Binary vars (for %): {len(BIN_VARS)}")

# Optional: sanity check cohort labels
if COHORT_COL in episodes.columns:
    print("\nCohort labels:")
    display(episodes[COHORT_COL].value_counts(dropna=False))


## 0) Sanity checks, analysis cohort, and sensitivity datasets

This section makes the analysis reproducible and consistent across **descriptives**, **Chi-square**, **Log-rank/KM**, and **Cox**:
- Ensures consistent filtering (time/event validity) and reports *N included*.
- Checks cohort labels (to avoid silent split/merge due to typos/whitespace).
- Prepares optional sensitivity datasets:
  - **First episode per patient**
  - **Landmark** (to mitigate time-dependent definition of coronal restoration window)


In [ ]:
# -----------------------
# 0) Sanity checks + consistent analysis cohort
# -----------------------

COHORT_COL = COHORT_COL if "COHORT_COL" in globals() else "Cohort"
ID_COL = ID_COL if "ID_COL" in globals() else "Patient_ID"
TIME_COL = TIME_COL if "TIME_COL" in globals() else "duration_days"
EVENT_COL = EVENT_COL if "EVENT_COL" in globals() else "event"

# --- Cohort label sanity ---
if COHORT_COL in episodes.columns:
    print("Cohort distribution (raw labels):")
    display(episodes[COHORT_COL].value_counts(dropna=False))

    # Common cleaning: strip whitespace
    episodes[COHORT_COL] = episodes[COHORT_COL].astype(str).str.strip().replace({"nan": np.nan})

    print("Cohort distribution (after strip):")
    display(episodes[COHORT_COL].value_counts(dropna=False))

# --- Consistent base filtering for ALL sections ---
episodes_base = episodes.copy()
episodes_base[TIME_COL] = pd.to_numeric(episodes_base.get(TIME_COL), errors="coerce")
episodes_base[EVENT_COL] = pd.to_numeric(episodes_base.get(EVENT_COL), errors="coerce").fillna(0).astype(int)

episodes_base = episodes_base.dropna(subset=[TIME_COL, EVENT_COL])
episodes_base = episodes_base[episodes_base[TIME_COL] >= 0]

def analysis_counts(df, label):
    return pd.Series({
        "Label": label,
        "Episodes (n)": int(len(df)),
        "Unique patients (n)": int(df[ID_COL].nunique()) if ID_COL in df.columns else np.nan,
        "Failures (n)": int(df[EVENT_COL].sum()) if EVENT_COL in df.columns else np.nan,
        "Failure rate (%)": float(100*df[EVENT_COL].mean()) if len(df) else np.nan,
        "Median follow-up (days)": float(df[TIME_COL].median()) if TIME_COL in df.columns else np.nan,
    })

print("Analysis cohort (base filters) counts:")
display(analysis_counts(episodes_base, "episodes_base"))

# Overwrite episodes used downstream to avoid section-to-section drift
episodes = episodes_base

# --- Sensitivity dataset: first episode per patient ---
if ID_COL in episodes.columns:
    first_episode_df = (episodes.sort_values(TIME_COL).groupby(ID_COL, as_index=False).first())
    print("Sensitivity: first episode per patient counts:")
    display(analysis_counts(first_episode_df, "first_episode_df"))

# --- Sensitivity dataset: landmark (time-dependent exposure concern) ---
# If coronal restoration group is defined using treatments occurring AFTER time zero,
# consider landmarking at LANDMARK_DAYS: exclude failures before landmark and reset time origin.
LANDMARK_DAYS = globals().get("LANDMARK_DAYS", None)
if LANDMARK_DAYS is None:
    # default: use WINDOW_DAYS if defined, else 90 days
    LANDMARK_DAYS = int(globals().get("WINDOW_DAYS", 90))
print(f"LANDMARK_DAYS (editable): {LANDMARK_DAYS}")

landmark_df = episodes.copy()
landmark_df = landmark_df[landmark_df[TIME_COL] > LANDMARK_DAYS].copy()
landmark_df[TIME_COL] = landmark_df[TIME_COL] - LANDMARK_DAYS

print("Sensitivity: landmark_df counts:")
display(analysis_counts(landmark_df, "landmark_df"))

# --- Multiple testing helper (BH/FDR) ---
def fdr_bh(pvals):
    p = np.array([np.nan if v is None else v for v in pvals], dtype=float)
    out = np.full_like(p, np.nan)
    m = np.sum(~np.isnan(p))
    if m == 0:
        return out
    idx = np.argsort(p, kind="mergesort")
    ranked = p[idx]
    # ignore nans at end
    valid = ~np.isnan(ranked)
    idx_valid = idx[valid]
    ranked_valid = ranked[valid]
    q = ranked_valid * m / (np.arange(1, len(ranked_valid)+1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out[idx_valid] = q
    return out


## Reviewer comment #73 — Follow-up time distribution (Primary RCT cohort)

Reports mean ± SD, median (IQR), and number of episodes per follow-up interval (< 1 yr, 1–3 yr, 3–5 yr, > 5 yr), as requested.

In [ ]:
# ---- Reviewer comment #73: Follow-up time distribution ----
# Adds mean±SD, median (IQR), and distribution by follow-up interval

rct = episodes[episodes['Cohort'] == 'Root canal treatment'].copy()

fu = rct['duration_days']
mean_fu = fu.mean()
std_fu  = fu.std()
med_fu  = fu.median()
q1_fu   = fu.quantile(0.25)
q3_fu   = fu.quantile(0.75)

print('Follow-up summary — Primary RCT cohort:')
print(f'  Mean \u00b1 SD       : {mean_fu:.1f} \u00b1 {std_fu:.1f} days')
print(f'  Median (IQR)    : {med_fu:.1f} ({q1_fu:.1f}\u2013{q3_fu:.1f}) days')

# Distribution by time interval
bins   = [0, 365, 3*365, 5*365, float('inf')]
labels = ['<1 year', '1\u20133 years', '3\u20135 years', '>5 years']
rct['fu_group'] = pd.cut(rct['duration_days'], bins=bins, labels=labels, right=False)

fu_dist = (
    rct.groupby('fu_group', observed=True)
       .agg(Episodes=('event', 'count'), Failures=('event', 'sum'))
       .reset_index()
)
fu_dist['Failure rate (%)'] = (fu_dist['Failures'] / fu_dist['Episodes'] * 100).round(2)
print('\nEpisodes by follow-up interval:')
display(fu_dist)


In [ ]:
# -----------------------

# PROBE 2023 audit - primary root canal treatment cohort

# -----------------------

# Purpose: build a manuscript-ready audit for cohort flow, exclusions, follow-up,

# missing data, and model complete-case availability using the finalized objects

# from this notebook only.



expected_counts = {

    "episodes": 119762,

    "patients": 87185,

    "failures": 3490,

    "censored": 116272,

}



expected_resto_groups = ["Neither", "Sealing only", "Sealing + Crown"]

requested_default_model_vars = [

    "duration_days",

    "event",

    "Coronal_Restoration_Group",

    "AgeGroup",

    "Male",

    "Smoking",

    "Cancer",

    "Diabetes",

    "Hypertension",

    "Biphos_use",

]



def _has_cols(df, cols):

    return all(col in df.columns for col in cols)



def _safe_nunique(df, col):

    return int(df[col].dropna().nunique()) if col in df.columns else np.nan



def _safe_sum(series_like):

    s = pd.to_numeric(series_like, errors="coerce")

    return int(s.fillna(0).sum()) if len(s) else 0



def _safe_missing(series_like):

    return int(pd.isna(series_like).sum())



def _fmt_pct(n, denom):

    if denom in [None, 0] or pd.isna(denom):

        return np.nan

    return round(float(100.0 * n / denom), 2)



def _series_or_na(df, col):

    if col in df.columns:

        return df[col]

    return pd.Series([np.nan] * len(df), index=df.index, dtype="object")



def _ordered_unique(values):

    out = []

    for value in values:

        if value not in out:

            out.append(value)

    return out



def _derive_rct_from_episodes(episodes_df):

    if not isinstance(episodes_df, pd.DataFrame) or "Cohort" not in episodes_df.columns:

        return pd.DataFrame()

    cohort_labels = episodes_df["Cohort"].astype(str).str.strip()

    return episodes_df.loc[cohort_labels == "Root canal treatment"].copy()



def _rct_key_set(df):

    candidate_keys = [

        ["episode_id"],

        ["Patient_ID", "Tooth_Num", "Cohort", "start_date"],

        ["Patient_ID", "Tooth_Num", "Cohort"],

    ]

    for key_cols in candidate_keys:

        if _has_cols(df, key_cols):

            return set(

                map(

                    tuple,

                    df[key_cols].astype(str).fillna("<NA>").drop_duplicates().itertuples(index=False, name=None),

                )

            )

    return None



def _resolve_rct(episodes_df):

    derived = _derive_rct_from_episodes(episodes_df)

    existing = globals().get("rct", None)



    if not isinstance(existing, pd.DataFrame) or existing.empty:

        return derived, "rederived from episodes"



    if "Cohort" not in existing.columns:

        return derived, "rederived from episodes (existing rct missing Cohort)"



    existing_candidate = existing.copy()

    existing_candidate["Cohort"] = existing_candidate["Cohort"].astype(str).str.strip()

    if (existing_candidate["Cohort"] != "Root canal treatment").any():

        return derived, "rederived from episodes (existing rct contains non-RCT rows)"



    if derived.empty:

        return existing_candidate, "used existing rct (episodes subset unavailable)"



    existing_keys = _rct_key_set(existing_candidate)

    derived_keys = _rct_key_set(derived)

    if existing_keys is not None and derived_keys is not None and existing_keys == derived_keys:

        return existing_candidate, "used validated existing rct"



    if len(existing_candidate) == len(derived):

        return existing_candidate, "used existing rct (matched derived row count)"



    return derived, "rederived from episodes (existing rct did not match derived subset)"



def _resolve_model_vars(rct_df):

    candidate_specs = []



    covars_rct_value = globals().get("covars_rct", None)

    if isinstance(covars_rct_value, list):

        candidate_specs.append((

            "covars_rct + survival vars",

            [TIME_COL, EVENT_COL] + covars_rct_value,

        ))



    base_covars_value = globals().get("base_covars", None)

    if isinstance(base_covars_value, list):

        candidate_specs.append((

            "base_covars + survival vars",

            [TIME_COL, EVENT_COL] + [col for col in base_covars_value if col != "Cohort"],

        ))



    candidate_specs.append(("requested default variable list", requested_default_model_vars))



    for basis, candidate_vars in candidate_specs:

        ordered = _ordered_unique(candidate_vars)

        available = [col for col in ordered if col in rct_df.columns]

        if available:

            return basis, ordered, available



    return "no model variables available", requested_default_model_vars, []



def _fallback_assign_cohort(text):

    t = str(text)

    tl = t.lower()

    if "אפיס" in t or "אפיסקט" in t or "apic" in tl:

        return "Apicoectomy"

    if "חידוש" in t:

        return "Root canal retreatment"

    if "טיפול שורש" in t:

        return "Root canal treatment"

    return None



def _prepare_raw_for_probe(raw_df):

    if not isinstance(raw_df, pd.DataFrame):

        return pd.DataFrame(), []



    raw_local = raw_df.copy()

    prep_notes = []



    if "Treatment_Date_dt" not in raw_local.columns and "Treatment_Date" in raw_local.columns:

        raw_local["Treatment_Date_dt"] = pd.to_datetime(raw_local["Treatment_Date"], errors="coerce", dayfirst=True)

        prep_notes.append("derived Treatment_Date_dt from Treatment_Date")



    if "Failure_Treatment_Date_dt" not in raw_local.columns and "Failure_Treatment_Date" in raw_local.columns:

        raw_local["Failure_Treatment_Date_dt"] = pd.to_datetime(raw_local["Failure_Treatment_Date"], errors="coerce", dayfirst=True)

        prep_notes.append("derived Failure_Treatment_Date_dt from Failure_Treatment_Date")



    assign_cohort_fn = globals().get("assign_cohort", None)

    if not callable(assign_cohort_fn):

        assign_cohort_fn = _fallback_assign_cohort



    if "סיווג" in raw_local.columns:

        derived_cohort = raw_local["סיווג"].fillna("").astype(str).apply(assign_cohort_fn)

        if "Cohort" not in raw_local.columns:

            raw_local["Cohort"] = derived_cohort

            prep_notes.append("derived Cohort from סיווג")

        elif raw_local["Cohort"].isna().any():

            raw_local["Cohort"] = raw_local["Cohort"].where(raw_local["Cohort"].notna(), derived_cohort)

            prep_notes.append("filled missing Cohort values from סיווג")



    return raw_local, prep_notes



def _rebuild_prefilter_episode_frame(raw_df, study_end_value):

    required = ["Patient_ID", "Tooth_Num", "Treatment_Date_dt", "Cohort"]

    if not isinstance(raw_df, pd.DataFrame) or not _has_cols(raw_df, required):

        return pd.DataFrame()



    pre_rows = raw_df.dropna(subset=required).copy()

    if pre_rows.empty:

        return pd.DataFrame()



    starts_local = (

        pre_rows.groupby(["Patient_ID", "Tooth_Num", "Cohort"], as_index=False)

        .agg(start_date=("Treatment_Date_dt", "min"))

    )



    if "Failure_Treatment_Date_dt" in pre_rows.columns:

        fail_tmp = pre_rows.merge(

            starts_local,

            on=["Patient_ID", "Tooth_Num", "Cohort"],

            how="inner",

        )

        fail_tmp = fail_tmp[fail_tmp["Failure_Treatment_Date_dt"].notna()].copy()

        fail_tmp = fail_tmp[fail_tmp["Failure_Treatment_Date_dt"] >= fail_tmp["start_date"]].copy()

        min_fail_local = (

            fail_tmp.groupby(["Patient_ID", "Tooth_Num", "Cohort"], as_index=False)

            .agg(failure_date=("Failure_Treatment_Date_dt", "min"))

        )

    else:

        min_fail_local = starts_local[["Patient_ID", "Tooth_Num", "Cohort"]].copy()

        min_fail_local["failure_date"] = pd.NaT



    prefilter = starts_local.merge(

        min_fail_local,

        on=["Patient_ID", "Tooth_Num", "Cohort"],

        how="left",

    )

    prefilter["event"] = prefilter["failure_date"].notna().astype(int)

    prefilter["stop_date"] = pd.to_datetime(

        prefilter["failure_date"].fillna(study_end_value),

        errors="coerce",

    )

    prefilter["duration_days"] = (

        pd.to_datetime(prefilter["stop_date"], errors="coerce")

        - pd.to_datetime(prefilter["start_date"], errors="coerce")

    ).dt.days

    return prefilter



if "episodes" not in globals() or not isinstance(episodes, pd.DataFrame):

    raise RuntimeError("The PROBE audit requires the finalized `episodes` DataFrame from earlier cells.")



if "raw" not in globals() or not isinstance(raw, pd.DataFrame):

    raise RuntimeError("The PROBE audit requires the `raw` DataFrame from earlier cells.")



raw_for_probe, raw_prep_notes = _prepare_raw_for_probe(raw)

rct, rct_source_note = _resolve_rct(episodes)

if rct.empty:

    raise RuntimeError("Could not derive the primary RCT cohort from `episodes`.")



if "event" in rct.columns:

    rct["event"] = pd.to_numeric(rct["event"], errors="coerce")

if "duration_days" in rct.columns:

    rct["duration_days"] = pd.to_numeric(rct["duration_days"], errors="coerce")



rct_episode_n = int(len(rct))

rct_patient_n = _safe_nunique(rct, "Patient_ID")

if _has_cols(rct, ["Patient_ID", "Tooth_Num"]):

    rct_unique_patient_tooth_n = int(rct[["Patient_ID", "Tooth_Num"]].drop_duplicates().shape[0])

else:

    rct_unique_patient_tooth_n = np.nan

rct_failure_n = _safe_sum(_series_or_na(rct, "event") == 1)

rct_censored_n = _safe_sum(_series_or_na(rct, "event") == 0)



print("PROBE audit - primary RCT analytical cohort")

print(f"  RCT source                          : {rct_source_note}")

if raw_prep_notes:

    print(f"  Raw audit preparation               : {'; '.join(raw_prep_notes)}")

print(f"  Primary RCT episodes                 : {rct_episode_n:,}")

print(f"  Unique patients                     : {rct_patient_n:,}" if not pd.isna(rct_patient_n) else "  Unique patients                     : [not available]")

print(f"  Unique Patient_ID + Tooth_Num pairs : {rct_unique_patient_tooth_n:,}" if not pd.isna(rct_unique_patient_tooth_n) else "  Unique Patient_ID + Tooth_Num pairs : [not available]")

print(f"  Failures / extractions (event = 1)  : {rct_failure_n:,}")

print(f"  Censored / survived (event = 0)     : {rct_censored_n:,}")



if rct_episode_n != expected_counts["episodes"]:

    print(f"WARNING: primary RCT episodes differ from manuscript expectation ({rct_episode_n:,} vs {expected_counts['episodes']:,}).")

if (not pd.isna(rct_patient_n)) and rct_patient_n != expected_counts["patients"]:

    print(f"WARNING: primary RCT patients differ from manuscript expectation ({rct_patient_n:,} vs {expected_counts['patients']:,}).")

if rct_failure_n != expected_counts["failures"]:

    print(f"WARNING: primary RCT failures differ from manuscript expectation ({rct_failure_n:,} vs {expected_counts['failures']:,}).")

if rct_censored_n != expected_counts["censored"]:

    print(f"WARNING: primary RCT censored/survived differ from manuscript expectation ({rct_censored_n:,} vs {expected_counts['censored']:,}).")



study_end_value = globals().get("STUDY_END", pd.NaT)

prefilter_episodes = _rebuild_prefilter_episode_frame(raw_for_probe, study_end_value)



eligible_required = ["Patient_ID", "Tooth_Num", "Treatment_Date_dt", "Cohort"]

raw_rows_n = int(len(raw_for_probe))

raw_eligible_rows_n = int(raw_for_probe.dropna(subset=[c for c in eligible_required if c in raw_for_probe.columns]).shape[0]) if _has_cols(raw_for_probe, eligible_required) else np.nan

if _has_cols(raw_for_probe, eligible_required):

    eligible_rows = raw_for_probe.dropna(subset=eligible_required).copy()

    pre_episode_combo_n = int(eligible_rows[["Patient_ID", "Tooth_Num", "Cohort"]].drop_duplicates().shape[0])

else:

    eligible_rows = pd.DataFrame()

    pre_episode_combo_n = np.nan



cohort_flow = pd.DataFrame([

    {

        "Stage": "Raw rows in source file",

        "n": raw_rows_n,

        "Notes": "Row-level records read from the source Excel file.",

    },

    {

        "Stage": "Rows eligible for episode construction",

        "n": raw_eligible_rows_n,

        "Notes": "Rows with non-missing Patient_ID, Tooth_Num, Treatment_Date_dt, and Cohort.",

    },

    {

        "Stage": "Unique Patient_ID × Tooth_Num × Cohort combinations before follow-up filtering",

        "n": pre_episode_combo_n,

        "Notes": "Episode keys before duration-based filtering.",

    },

    {

        "Stage": "Episode-level records after duration/follow-up filtering",

        "n": int(len(episodes)),

        "Notes": "Final analytic episode dataset after dropping missing or negative duration_days.",

    },

    {

        "Stage": "Primary RCT episodes",

        "n": rct_episode_n,

        "Notes": "Episodes with Cohort = Root canal treatment.",

    },

    {

        "Stage": "Primary RCT failures/extractions",

        "n": rct_failure_n,

        "Notes": "Primary RCT episodes with event = 1.",

    },

    {

        "Stage": "Primary RCT censored/survived",

        "n": rct_censored_n,

        "Notes": "Primary RCT episodes with event = 0.",

    },

])



model_basis, model_vars_requested, model_vars_available = _resolve_model_vars(rct)

model_missing_any = int(rct[model_vars_available].isna().any(axis=1).sum()) if model_vars_available else np.nan



diagnostic_rows = []



raw_denominator = int(len(raw_for_probe))

for col in ["Patient_ID", "Tooth_Num", "Treatment_Date_dt", "Cohort"]:

    n_missing = _safe_missing(_series_or_na(raw_for_probe, col))

    diagnostic_rows.append({

        "Check": f"Raw rows missing {col}",

        "n": n_missing,

        "Denominator": raw_denominator,

        "Interpretation": f"Source-row diagnostic before cohort construction for {col}, using current notebook-consistent audit fields.",

        "True exclusion yes/no/unknown": "unknown",

    })



prefilter_denominator = int(len(prefilter_episodes))

prefilter_duration_missing_n = _safe_missing(_series_or_na(prefilter_episodes, "duration_days")) if prefilter_denominator else np.nan

prefilter_duration_negative_n = int((pd.to_numeric(_series_or_na(prefilter_episodes, "duration_days"), errors="coerce") < 0).fillna(False).sum()) if prefilter_denominator else np.nan

diagnostic_rows.append({

    "Check": "Episode-level records missing duration_days before duration filtering",

    "n": prefilter_duration_missing_n,

    "Denominator": prefilter_denominator,

    "Interpretation": "Prefilter episode diagnostic reconstructed from the current raw-to-episode logic.",

    "True exclusion yes/no/unknown": "yes",

})

diagnostic_rows.append({

    "Check": "Episode-level records with duration_days < 0 before duration filtering",

    "n": prefilter_duration_negative_n,

    "Denominator": prefilter_denominator,

    "Interpretation": "Prefilter episode diagnostic reconstructed from the current raw-to-episode logic.",

    "True exclusion yes/no/unknown": "yes",

})



rct_denominator = int(len(rct))

rct_duration_missing_n = _safe_missing(_series_or_na(rct, "duration_days"))

rct_duration_negative_n = int((pd.to_numeric(_series_or_na(rct, "duration_days"), errors="coerce") < 0).fillna(False).sum())

failure_mask = pd.to_numeric(_series_or_na(rct, "event"), errors="coerce") == 1

censor_mask = pd.to_numeric(_series_or_na(rct, "event"), errors="coerce") == 0

failure_denominator = int(failure_mask.sum())

censor_denominator = int(censor_mask.sum())

failure_date_missing_in_failures_n = _safe_missing(_series_or_na(rct.loc[failure_mask], "failure_date")) if failure_denominator else np.nan

stop_date_missing_in_censored_n = _safe_missing(_series_or_na(rct.loc[censor_mask], "stop_date")) if censor_denominator else np.nan



if "Coronal_Restoration_Group" in rct.columns:

    resto_missing_or_unclassified_n = int((

        rct["Coronal_Restoration_Group"].isna()

        | ~rct["Coronal_Restoration_Group"].isin(expected_resto_groups)

    ).sum())

else:

    resto_missing_or_unclassified_n = np.nan



diagnostic_rows.extend([

    {

        "Check": "Primary RCT episodes missing duration_days",

        "n": rct_duration_missing_n,

        "Denominator": rct_denominator,

        "Interpretation": "Diagnostic within the final analytical cohort.",

        "True exclusion yes/no/unknown": "no",

    },

    {

        "Check": "Primary RCT episodes with duration_days < 0",

        "n": rct_duration_negative_n,

        "Denominator": rct_denominator,

        "Interpretation": "Diagnostic within the final analytical cohort; should be zero after filtering.",

        "True exclusion yes/no/unknown": "no",

    },

    {

        "Check": "Failures/extractions missing failure_date",

        "n": failure_date_missing_in_failures_n,

        "Denominator": failure_denominator,

        "Interpretation": "Assessed only among event = 1 episodes; failure_date is structurally required here.",

        "True exclusion yes/no/unknown": "unknown",

    },

    {

        "Check": "Censored episodes missing stop_date",

        "n": stop_date_missing_in_censored_n,

        "Denominator": censor_denominator,

        "Interpretation": "Assessed only among event = 0 episodes; stop_date should reflect censoring date.",

        "True exclusion yes/no/unknown": "unknown",

    },

    {

        "Check": "Missing or unclassifiable Coronal_Restoration_Group",

        "n": resto_missing_or_unclassified_n,

        "Denominator": rct_denominator,

        "Interpretation": "Diagnostic classification check for the restoration grouping variable.",

        "True exclusion yes/no/unknown": "unknown",

    },

    {

        "Check": "Primary RCT episodes missing at least one model covariate",

        "n": model_missing_any,

        "Denominator": rct_denominator,

        "Interpretation": f"True exclusion from a complete-case multivariable model using {model_basis}.",

        "True exclusion yes/no/unknown": "yes",

    },

])



diagnostic_exclusions = pd.DataFrame(diagnostic_rows)



missing_specs = [

    ("Core/survival", "Patient_ID", None, "Analytical cohort identifier."),

    ("Core/survival", "Tooth_Num", None, "Analytical cohort tooth identifier."),

    ("Core/survival", "Cohort", None, "Cohort should be fixed to primary root canal treatment in this subset."),

    ("Core/survival", "start_date", None, "Episode start date."),

    ("Core/survival", "stop_date", None, "Required for both failures and censored episodes in the analytical cohort."),

    ("Core/survival", "duration_days", None, "Primary follow-up time variable."),

    ("Core/survival", "event", None, "Binary event indicator."),

    ("Core/survival", "failure_date", "event == 1", "Assessed only among failures/extractions; not required for censored teeth."),

    ("Patient/descriptive/model", "Age", None, "Age at treatment, if available."),

    ("Patient/descriptive/model", "AgeGroup", None, "Derived age group used in descriptive/model tables, if available."),

    ("Patient/descriptive/model", "Male", None, "Sex indicator, if available."),

    ("Patient/descriptive/model", "Smoking", None, "Patient-level covariate, if available."),

    ("Patient/descriptive/model", "Cancer", None, "Patient-level covariate, if available."),

    ("Patient/descriptive/model", "Diabetes", None, "Patient-level covariate, if available."),

    ("Patient/descriptive/model", "Hypertension", None, "Patient-level covariate, if available."),

    ("Patient/descriptive/model", "Biphos_use", None, "Patient-level covariate, if available."),

    ("Restoration", "Has_Sealing_in_window", None, "Absence of sealing is coded as 0 and is not missing."),

    ("Restoration", "Has_Crown_in_window", None, "Absence of crown is coded as 0 and is not missing."),

    ("Restoration", "Coronal_Restoration_Group", None, "Expected groups are Neither, Sealing only, and Sealing + Crown."),

]



missing_rows = []

for section, variable, condition, note in missing_specs:

    if variable not in rct.columns:

        missing_rows.append({

            "Section": section,

            "Variable": variable,

            "Denominator": np.nan,

            "Missing n": np.nan,

            "Missing %": np.nan,

            "Notes": f"Column not present in analytical cohort. {note}",

        })

        continue



    if condition == "event == 1":

        sub = rct.loc[failure_mask].copy()

        denominator = int(len(sub))

    else:

        sub = rct

        denominator = int(len(sub))



    missing_n = _safe_missing(sub[variable])

    missing_rows.append({

        "Section": section,

        "Variable": variable,

        "Denominator": denominator,

        "Missing n": missing_n,

        "Missing %": _fmt_pct(missing_n, denominator),

        "Notes": note,

    })



missing_data = pd.DataFrame(missing_rows)



restoration_rows = []

if "Coronal_Restoration_Group" in rct.columns:

    resto_counts = rct["Coronal_Restoration_Group"].value_counts(dropna=False)

    for group_name in expected_resto_groups:

        count = int(resto_counts.get(group_name, 0))

        restoration_rows.append({

            "Coronal_Restoration_Group": group_name,

            "n": count,

            "%": _fmt_pct(count, rct_denominator),

        })

    missing_resto_n = int(rct["Coronal_Restoration_Group"].isna().sum())

    unclassified_resto_n = int((

        rct["Coronal_Restoration_Group"].notna()

        & ~rct["Coronal_Restoration_Group"].isin(expected_resto_groups)

    ).sum())

    restoration_rows.append({

        "Coronal_Restoration_Group": "Missing",

        "n": missing_resto_n,

        "%": _fmt_pct(missing_resto_n, rct_denominator),

    })

    restoration_rows.append({

        "Coronal_Restoration_Group": "Unclassifiable",

        "n": unclassified_resto_n,

        "%": _fmt_pct(unclassified_resto_n, rct_denominator),

    })

else:

    restoration_rows.append({

        "Coronal_Restoration_Group": "Column not present",

        "n": np.nan,

        "%": np.nan,

    })



restoration_counts = pd.DataFrame(restoration_rows)



duration_series = pd.to_numeric(_series_or_na(rct, "duration_days"), errors="coerce")

followup_checks = pd.DataFrame([

    {"Metric": "Mean duration_days", "Value": duration_series.mean()},

    {"Metric": "SD duration_days", "Value": duration_series.std()},

    {"Metric": "Median duration_days", "Value": duration_series.median()},

    {"Metric": "IQR duration_days", "Value": duration_series.quantile(0.75) - duration_series.quantile(0.25)},

    {"Metric": "Min duration_days", "Value": duration_series.min()},

    {"Metric": "Max duration_days", "Value": duration_series.max()},

    {"Metric": "Missing duration_days", "Value": int(duration_series.isna().sum())},

    {"Metric": "Zero duration_days", "Value": int((duration_series == 0).fillna(False).sum())},

    {"Metric": "Negative duration_days", "Value": int((duration_series < 0).fillna(False).sum())},

    {"Metric": "Failures missing failure_date", "Value": failure_date_missing_in_failures_n},

    {"Metric": "Censored episodes missing stop_date", "Value": stop_date_missing_in_censored_n},

])



print("\nCohort-flow table:")

display(cohort_flow)



print("Diagnostic exclusions table:")

display(diagnostic_exclusions)



print("PROBE missing-data table:")

display(missing_data)



print("Restoration-group counts:")

display(restoration_counts)



print("Follow-up checks:")

display(followup_checks)



if "STUDY_END" in globals():

    print(f"STUDY_END in notebook: {pd.Timestamp(STUDY_END).date()}")

    if pd.Timestamp(STUDY_END) == pd.Timestamp("2021-01-01"):

        print("WARNING: STUDY_END is 2021-01-01. Verify that manuscript study period and follow-up wording match this data cutoff.")

else:

    print("STUDY_END is not defined in the current kernel state.")



complete_case_mask = ~rct[model_vars_available].isna().any(axis=1) if model_vars_available else pd.Series([True] * len(rct), index=rct.index)

model_complete_n = int(complete_case_mask.sum())

model_excluded_n = int((~complete_case_mask).sum())



model_complete_case_table = pd.DataFrame([

    {"Metric": "Primary RCT episodes", "n": rct_denominator},

    {"Metric": "Episodes complete for all model variables", "n": model_complete_n},

    {"Metric": "Episodes excluded from model due to missing model variables", "n": model_excluded_n},

])



model_missing_by_var_rows = []

for variable in model_vars_requested:

    if variable in rct.columns:

        missing_n = _safe_missing(rct[variable])

        denominator = rct_denominator

        note = f"Included in the complete-case audit ({model_basis})." if variable in model_vars_available else f"Available in cohort but not selected for the complete-case audit ({model_basis})."

    else:

        missing_n = np.nan

        denominator = np.nan

        note = f"Column not present in analytical cohort; not counted as missing complete-case data ({model_basis})."

    model_missing_by_var_rows.append({

        "Variable": variable,

        "Missing n": missing_n,

        "Missing %": _fmt_pct(missing_n, denominator),

        "Denominator": denominator,

        "Notes": note,

    })



model_missing_by_var = pd.DataFrame(model_missing_by_var_rows)



print(f"Model audit basis: {model_basis}")

print(f"Model variables requested: {model_vars_requested}")

print(f"Model variables available: {model_vars_available}")

print("Model complete-case audit:")

display(model_complete_case_table)



print("Model missingness by variable:")

display(model_missing_by_var)



output_path = os.path.join(os.getcwd(), "probe_audit_primary_rct.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    cohort_flow.to_excel(writer, sheet_name="cohort_flow", index=False)

    diagnostic_exclusions.to_excel(writer, sheet_name="diagnostic_exclusions", index=False)

    missing_data.to_excel(writer, sheet_name="missing_data", index=False)

    restoration_counts.to_excel(writer, sheet_name="restoration_counts", index=False)

    followup_checks.to_excel(writer, sheet_name="followup_checks", index=False)

    model_complete_case_table.to_excel(writer, sheet_name="model_complete_case", index=False)

    model_missing_by_var.to_excel(writer, sheet_name="model_missing_by_var", index=False)



print(f"Excel audit workbook written to: {output_path}")



methods_cohort_text = (

    f"Methods - cohort construction: The analytical cohort comprised {rct_episode_n:,} primary root canal treatment episodes "

    f"identified from {raw_rows_n:,} source rows. After restricting to rows with non-missing Patient_ID, Tooth_Num, "

    f"Treatment_Date_dt, and Cohort ({raw_eligible_rows_n:,} rows), {pre_episode_combo_n:,} unique Patient_ID x Tooth_Num x Cohort "

    f"combinations were available before follow-up filtering. The final episode-level dataset contained {len(episodes):,} records, of which "

    f"{rct_episode_n:,} were primary root canal treatment episodes."

)



methods_missing_text = (

    f"Methods - missing-data handling: Missing-data assessment was performed on the analytical primary root canal treatment cohort, not on the full raw source table. "

    f"Core follow-up variables were audited within all {rct_episode_n:,} episodes, whereas failure_date was assessed conditionally among failures only "

    f"(n={failure_denominator:,}). Binary restoration indicators were treated as coded absence when equal to 0 and were not classified as missing. "

    f"For the multivariable complete-case audit, {model_complete_n:,} of {rct_denominator:,} episodes had complete data for all model variables available in the cohort using {model_basis}."

)



results_flow_text = (

    f"Results - participant/episode flow: The primary root canal treatment cohort included {rct_episode_n:,} episodes from {rct_patient_n:,} unique patients, "

    f"representing {rct_unique_patient_tooth_n:,} unique Patient_ID + Tooth_Num combinations. Among these episodes, {rct_failure_n:,} ended in extraction/failure "

    f"and {rct_censored_n:,} were censored/survived at the end of follow-up."

)



if model_vars_available:

    missing_statement_core = []

    for variable in ["Age", "AgeGroup", "Male", "Smoking", "Cancer", "Diabetes", "Hypertension", "Biphos_use"]:

        if variable in missing_data["Variable"].values:

            row = missing_data.loc[missing_data["Variable"] == variable].iloc[0]

            if not pd.isna(row["Missing n"]):

                missing_statement_core.append(f"{variable}: {int(row['Missing n']):,} missing ({row['Missing %']:.2f}%)")

    missing_summary_text = "; ".join(missing_statement_core) if missing_statement_core else "No optional descriptive/model columns were available for missing-data summarization."

else:

    missing_summary_text = "Model variables were not available for complete-case auditing in the current cohort."



results_missing_text = (

    f"Results - missing-data statement: Within the primary root canal treatment cohort, missingness in the core survival variables was minimal or absent after analytical filtering. "

    f"Among failures, {failure_date_missing_in_failures_n if not pd.isna(failure_date_missing_in_failures_n) else '[X]'} episodes were missing failure_date; among censored episodes, "

    f"{stop_date_missing_in_censored_n if not pd.isna(stop_date_missing_in_censored_n) else '[X]'} were missing stop_date. Complete-case exclusion for the multivariable model affected "

    f"{model_excluded_n:,} of {rct_denominator:,} episodes. Variable-specific missingness was as follows: {missing_summary_text}"

)



print("\nDraft manuscript text")

print(methods_cohort_text)

print()

print(methods_missing_text)

print()

print(results_flow_text)

print()

print(results_missing_text)


## Data structure, time windows, and analysis framework

### Episodes and cohorts
The unit of analysis is an episode, defined according to cohort-specific inclusion criteria (e.g., Apicoectomy, Root canal treatment, Root canal retreatment). Each episode is assigned to a single clinical cohort and contributes follow-up time independently.

### Time windows and exposure definition
Selected treatment-related covariates (e.g., Has_Sealing_in_window, Has_Crown_in_window) are defined based on their occurrence within a predefined time window relative to the episode index date. These variables indicate whether the corresponding procedure was performed during the specified window and are treated as binary (0/1) exposures in the analyses.

### Outcome and follow-up
The study outcome is a time-to-event endpoint. For each episode, follow-up begins at the episode index date and continues until outcome occurrence (failure) or censoring. Episodes without an observed failure contribute censored follow-up time.


## 1) Descriptive outcomes (success / failure) and population description


In [ ]:
def summarize_groups(df, group_cols, id_col="Patient_ID", time_col="duration_days", event_col="event"):
    """Episode-level descriptive summary.

    - If group_cols is empty, returns a single-row 'Overall' summary (prevents pandas groupby([]) errors).
    """
    d = df.copy()
    d[time_col] = pd.to_numeric(d.get(time_col), errors="coerce")
    d[event_col] = pd.to_numeric(d.get(event_col), errors="coerce").fillna(0).astype(int)

    def _one_group(g):
        return pd.Series({
            "Episodes (n)": len(g),
            "Unique patients (n)": g[id_col].dropna().nunique() if id_col in g.columns else np.nan,
            "Failures (n)": int(g[event_col].sum()) if event_col in g.columns else np.nan,
            "Failure rate (%)": 100.0 * g[event_col].mean() if len(g) and event_col in g.columns else np.nan,
            "Success rate (%)": 100.0 * (1 - g[event_col].mean()) if len(g) and event_col in g.columns else np.nan,
            "Follow-up (days) median": g[time_col].median() if time_col in g.columns else np.nan,
            "Follow-up (days) IQR": (g[time_col].quantile(0.75) - g[time_col].quantile(0.25)) if time_col in g.columns else np.nan,
        })

    # Overall (no grouping)
    if not group_cols:
        out = _one_group(d).to_frame().T
        out.insert(0, "Group", "Overall")
        return out

    # Grouped
    out = (d.groupby(group_cols, dropna=False).apply(_one_group)).reset_index()
    return out

def table1_population(df, group_col, systemic_cols=None):
    """Compact Table-1 style population description (episodes as rows)."""
    d = df.copy()
    d["Age"] = pd.to_numeric(d.get("Age"), errors="coerce")

    # Avoid reliance on a global; default to empty list
    if systemic_cols is None:
        systemic_cols = []
    cols = ["Age", "Male"] + [c for c in systemic_cols if c in d.columns]

    rows = []
    for name, g in d.groupby(group_col, dropna=False):
        r = {
            "Group": name,
            "Episodes (n)": len(g),
            "Patients (n)": g["Patient_ID"].nunique() if "Patient_ID" in g.columns else np.nan,
        }
        r["Age mean (SD)"] = f"{g['Age'].mean():.1f} ({g['Age'].std():.1f})" if g["Age"].notna().any() else ""
        r["Age median (IQR)"] = f"{g['Age'].median():.1f} ({(g['Age'].quantile(0.75)-g['Age'].quantile(0.25)):.1f})" if g["Age"].notna().any() else ""
        r["Female (%)"] = f"{100*(1-g['Male'].mean()):.1f}" if "Male" in g.columns and g["Male"].notna().any() else ""

        for c in [c for c in cols if c not in ["Age", "Male"]]:
            r[f"{c} (%)"] = f"{100*g[c].mean():.1f}" if c in g.columns and g[c].notna().any() else ""

        rows.append(r)

    return pd.DataFrame(rows)

print("Overall outcomes:")
display(summarize_groups(episodes, []))

print("By cohort:")
display(summarize_groups(episodes, ["Cohort"]))

print("By coronal restoration group:")
display(summarize_groups(episodes, ["Coronal_Restoration_Group"]))

print("By cohort × coronal restoration group:")
display(summarize_groups(episodes, ["Cohort","Coronal_Restoration_Group"]))

print("Population description (Table 1) by cohort:")
display(table1_population(episodes, "Cohort", systemic_cols=systemic_cols))

print("Population description (Table 1) by coronal restoration group:")
display(table1_population(episodes, "Coronal_Restoration_Group", systemic_cols=systemic_cols))


In [ ]:
# -----------------------
# Descriptive outcomes: overall + by key groupings + by ALL categorical variables
# -----------------------

print("Overall outcomes:")
display(summarize_groups(episodes, []))

print("By cohort (treatment type):")
display(summarize_groups(episodes, [COHORT_COL] if COHORT_COL in episodes.columns else ["Cohort"]))

# Optional: by coronal restoration group if exists
if "Coronal_Restoration_Group" in episodes.columns:
    print("By coronal restoration group:")
    display(summarize_groups(episodes, ["Coronal_Restoration_Group"]))

    if COHORT_COL in episodes.columns:
        print("By cohort × coronal restoration group:")
        display(summarize_groups(episodes, [COHORT_COL, "Coronal_Restoration_Group"]))

# Table 1 population description (episodes as rows) - by cohort
if "systemic_cols" in globals():
    sys_cols = systemic_cols
else:
    # fallback: use binary vars excluding Male if present
    sys_cols = [c for c in BIN_VARS if c not in ["Male"]]

print("Population description (Table 1) by cohort:")
if COHORT_COL in episodes.columns:
    display(table1_population(episodes, COHORT_COL, systemic_cols=sys_cols))

# -----
# Cox-style descriptive outcomes for ALL Cox covariates (binary / dummy-coded)
# -----
# Rationale:
# - In Cox, multi-level categoricals are represented by dummy variables relative to a reference category.
# - For clean reporting, we describe crude outcomes for covariate==0 (reference) vs covariate==1 (compared),
#   and we do this BOTH overall and within each cohort (treatment type).

def outcomes_like_cox(df, covariate_cols,
                      id_col=ID_COL, time_col=TIME_COL, event_col=EVENT_COL,
                      include_reference_row=True):
    d = df.copy()
    d[time_col] = pd.to_numeric(d.get(time_col), errors="coerce")
    d[event_col] = pd.to_numeric(d.get(event_col), errors="coerce").fillna(0).astype(int)

    def _summ(g):
        return {
            "Episodes (n)": int(len(g)),
            "Unique patients (n)": int(g[id_col].nunique()) if id_col in g.columns else np.nan,
            "Failures (n)": int(g[event_col].sum()) if event_col in g.columns else np.nan,
            "Failure rate (%)": float(100.0 * g[event_col].mean()) if len(g) else np.nan,
            "Success rate (%)": float(100.0 * (1 - g[event_col].mean())) if len(g) else np.nan,
            "Follow-up (days) median": float(g[time_col].median()) if time_col in g.columns else np.nan,
            "Follow-up (days) IQR": float(g[time_col].quantile(0.75) - g[time_col].quantile(0.25)) if time_col in g.columns else np.nan,
        }

    rows = []
    base = d.dropna(subset=[event_col])

    for v in covariate_cols:
        if v not in base.columns:
            rows.append({"Variable": v, "Level": None, "note": "missing column in df"})
            continue

        x = pd.to_numeric(base[v], errors="coerce")
        ok = x.isin([0, 1]) & base[event_col].notna()

        g0 = base.loc[ok & (x == 0)]
        g1 = base.loc[ok & (x == 1)]

        if len(g0) == 0 and len(g1) == 0:
            rows.append({"Variable": v, "Level": None, "note": "no valid 0/1 values"})
            continue

        if include_reference_row:
            r0 = {"Variable": v, "Level": "Reference (0)"}
            r0.update(_summ(g0))
            r0["note"] = ""
            rows.append(r0)

        r1 = {"Variable": v, "Level": "Compared (1)"}
        r1.update(_summ(g1))
        r1["note"] = ""
        rows.append(r1)

    out = pd.DataFrame(rows)
    core = ["Variable", "Level", "Episodes (n)", "Unique patients (n)", "Failures (n)",
            "Failure rate (%)", "Success rate (%)", "Follow-up (days) median",
            "Follow-up (days) IQR", "note"]
    for c in core:
        if c not in out.columns:
            out[c] = ""
    return out[core]

def outcomes_like_cox_within_groups(df, group_col, covariate_cols,
                                   id_col=ID_COL, time_col=TIME_COL, event_col=EVENT_COL,
                                   include_reference_row=True):
    d = df.copy()
    d[time_col] = pd.to_numeric(d.get(time_col), errors="coerce")
    d[event_col] = pd.to_numeric(d.get(event_col), errors="coerce").fillna(0).astype(int)

    rows = []
    for grp, dg in d.groupby(group_col, dropna=False):
        t = outcomes_like_cox(dg, covariate_cols,
                             id_col=id_col, time_col=time_col, event_col=event_col,
                             include_reference_row=include_reference_row)
        t.insert(0, group_col, grp)
        rows.append(t)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

print("Cox-style outcomes for ALL Cox covariates (overall):")
cox_style_overall = outcomes_like_cox(episodes, COX_VARS)
display(cox_style_overall)

if COHORT_COL in episodes.columns:
    print("Cox-style outcomes for ALL Cox covariates WITHIN each cohort:")
    cox_style_within = outcomes_like_cox_within_groups(episodes, COHORT_COL, COX_VARS)
    display(cox_style_within)


## Reviewer comments #76 + #92 — Detailed Table 1

Replaces the simple episode-count table with a full demographics table (absolute numbers + %, age mean±SD & median IQR, tooth location, restoration type), as requested.

In [ ]:
# ---- Reviewer comments #76 + #92: Detailed Table 1 ----
# Full demographics + tooth location + restoration type, absolute n and %

rct = episodes[episodes['Cohort'] == 'Root canal treatment'].copy()

# ── Tooth location (FDI two-digit numbering) ──────────────────────────
def classify_tooth(num):
    try:
        n = int(num)
    except (ValueError, TypeError):
        return 'Unknown', 'Unknown'
    quadrant      = n // 10
    tooth_in_quad = n % 10
    arch     = 'Upper'    if quadrant in [1, 2] else ('Lower' if quadrant in [3, 4] else 'Unknown')
    position = 'Anterior' if tooth_in_quad in [1, 2, 3] else ('Posterior' if tooth_in_quad in [4, 5, 6, 7, 8] else 'Unknown')
    return arch, position

rct[['Arch', 'Position']] = rct['Tooth_Num'].apply(lambda x: pd.Series(classify_tooth(x)))

# ── Resolve column names (_x/_y suffix from merge) ────────────────────
def resolve(df, name):
    for suffix in ['', '_x', '_y']:
        if name + suffix in df.columns:
            return name + suffix
    return None

age_col    = resolve(rct, 'Age')
male_col   = resolve(rct, 'Male')
smoke_col  = resolve(rct, 'Smoking')
cancer_col = resolve(rct, 'Cancer')
diab_col   = resolve(rct, 'Diabetes')
biphos_col = resolve(rct, 'Biphos_use')
hypert_col = resolve(rct, 'Hypertension')
agegrp_col = resolve(rct, 'AgeGroup')

n_total    = len(rct)
n_patients = rct['Patient_ID'].nunique() if 'Patient_ID' in rct.columns else float('nan')

rows = []
rows.append({'Characteristic': 'Episodes (n)',        'n': n_total,        '%': ''})
rows.append({'Characteristic': 'Unique patients (n)', 'n': int(n_patients), '%': ''})

# Age
if age_col:
    a = pd.to_numeric(rct[age_col], errors='coerce')
    rows.append({'Characteristic': 'Age, mean \u00b1 SD (years)',
                 'n': f'{a.mean():.1f} \u00b1 {a.std():.1f}', '%': ''})
    rows.append({'Characteristic': 'Age, median (IQR)',
                 'n': f'{a.median():.1f} ({a.quantile(0.25):.1f}\u2013{a.quantile(0.75):.1f})', '%': ''})

# Age groups
if agegrp_col:
    for ag in ['<40', '40\u201360', '\u226560']:
        candidates = [v for v in rct[agegrp_col].dropna().unique() if str(ag) in str(v) or ag == str(v)]
        for ag_val in candidates:
            mask = rct[agegrp_col] == ag_val
            rows.append({'Characteristic': f'  Age {ag_val}, n (%)',
                         'n': int(mask.sum()), '%': f'{100*mask.mean():.1f}'})

# Sex
if male_col:
    female = 1 - pd.to_numeric(rct[male_col], errors='coerce').fillna(0)
    rows.append({'Characteristic': 'Female, n (%)',
                 'n': int(female.sum()), '%': f'{100*female.mean():.1f}'})
    rows.append({'Characteristic': 'Male, n (%)',
                 'n': int(rct[male_col].sum()), '%': f"{100*rct[male_col].mean():.1f}"})

# Systemic conditions
for label, col in [
    ('Diabetes mellitus, n (%)',    diab_col),
    ('Smoking, n (%)',              smoke_col),
    ('Malignancy, n (%)',           cancer_col),
    ('Hypertension, n (%)',         hypert_col),
    ('Bisphosphonate use, n (%)',   biphos_col),
]:
    if col:
        s = pd.to_numeric(rct[col], errors='coerce').fillna(0)
        rows.append({'Characteristic': label, 'n': int(s.sum()), '%': f'{100*s.mean():.1f}'})

# Coronal restoration groups
rows.append({'Characteristic': '--- Restoration ---', 'n': '', '%': ''})
for rg in ['Sealing + Crown', 'Sealing only', 'Neither']:
    mask = rct['Coronal_Restoration_Group'] == rg
    rows.append({'Characteristic': f'  {rg}, n (%)',
                 'n': int(mask.sum()), '%': f'{100*mask.mean():.1f}'})

# Tooth location
rows.append({'Characteristic': '--- Tooth location ---', 'n': '', '%': ''})
for arch in ['Upper', 'Lower', 'Unknown']:
    mask = rct['Arch'] == arch
    if mask.sum() > 0:
        rows.append({'Characteristic': f'  Arch: {arch}, n (%)',
                     'n': int(mask.sum()), '%': f'{100*mask.mean():.1f}'})
for pos in ['Anterior', 'Posterior', 'Unknown']:
    mask = rct['Position'] == pos
    if mask.sum() > 0:
        rows.append({'Characteristic': f'  Position: {pos}, n (%)',
                     'n': int(mask.sum()), '%': f'{100*mask.mean():.1f}'})

# Outcomes
rows.append({'Characteristic': '--- Outcomes ---', 'n': '', '%': ''})
n_fail = int(rct['event'].sum())
rows.append({'Characteristic': 'Failures (extractions), n (%)',
             'n': n_fail, '%': f'{100*rct["event"].mean():.2f}'})
rows.append({'Characteristic': 'Survival (censored), n (%)',
             'n': n_total - n_fail, '%': f'{100*(1-rct["event"].mean()):.2f}'})

table1_detailed = pd.DataFrame(rows)[['Characteristic', 'n', '%']]
table1_detailed.columns = ['Characteristic', 'n  (or mean \u00b1 SD)', '% (or median IQR)']
print('Table 1 — Detailed demographics and tooth characteristics (Primary RCT, n = 119,762 episodes):')
display(table1_detailed)


## 2) Chi-square tests (categorical comparisons)


In [ ]:
# -----------------------
# 2) Chi-square / Fisher tests (proportion-based; ignores follow-up time)
# -----------------------
# Note: This tests differences in EVENT PROPORTIONS only (does NOT use follow-up time).

def chi2_or_fisher_2way(df, row, col, min_expected=5, dropna=True):
    """Return crosstab + test result with explicit labels.

    Returns:
      ct: contingency table
      res: dict with:
        - test (Chi-square or Fisher exact)
        - p_value (the p-value used for reporting)
        - p_chi2, chi2, df (if Chi-square computed)
        - p_fisher, odds_ratio (if Fisher)
        - min_expected (from chi-square expected counts)
    """
    d = df.copy()
    if dropna:
        d = d.dropna(subset=[row, col])

    ct = pd.crosstab(d[row], d[col], dropna=False)

    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return ct, None

    chi2, p_chi2, dof, exp = stats.chi2_contingency(ct)
    exp_min = float(np.min(exp))

    # Optional Fisher for 2x2 with sparse expected counts
    if ct.shape == (2, 2) and exp_min < min_expected:
        oddsratio, p_f = stats.fisher_exact(ct.values)
        return ct, {
            "row": row, "col": col,
            "test": "Fisher exact (2x2)",
            "p_value": float(p_f),
            "p_fisher": float(p_f),
            "odds_ratio": float(oddsratio),
            "p_chi2": float(p_chi2),
            "chi2": float(chi2),
            "df": int(dof),
            "min_expected": exp_min,
            "note": f"Fisher used (min expected {exp_min:.2f} < {min_expected})"
        }

    return ct, {
        "row": row, "col": col,
        "test": "Chi-square",
        "p_value": float(p_chi2),
        "p_chi2": float(p_chi2),
        "chi2": float(chi2),
        "df": int(dof),
        "p_fisher": np.nan,
        "odds_ratio": np.nan,
        "min_expected": exp_min,
        "note": "" if exp_min >= min_expected else f"Warning: min expected {exp_min:.2f} < {min_expected}"
    }


def run_chi2_batch(df, group_col, categorical_vars, event_col=EVENT_COL, min_expected=5):
    """Run chi2/fisher for each categorical var vs event_col within df."""
    rows = []
    for v in categorical_vars:
        if v not in df.columns:
            continue
        ct, res = chi2_or_fisher_2way(df, v, event_col, min_expected=min_expected, dropna=True)
        if res is None:
            rows.append({"Variable": v, "test": None, "p_value": np.nan, "note": "not testable"})
        else:
            rows.append({
                "Variable": v,
                "test": res["test"],
                "chi2": res.get("chi2", np.nan),
                "df": res.get("df", np.nan),
                "odds_ratio": res.get("odds_ratio", np.nan),
                "p_value": res.get("p_value", np.nan),
                "p_chi2": res.get("p_chi2", np.nan),
                "p_fisher": res.get("p_fisher", np.nan),
                "min_expected": res.get("min_expected", np.nan),
                "note": res.get("note","")
            })
    out = pd.DataFrame(rows)
    if not out.empty and "p_value" in out.columns:
        out["q_value_fdr_bh"] = fdr_bh(out["p_value"].values)
        out["p_value"] = out["p_value"].round(4)
        out["q_value_fdr_bh"] = out["q_value_fdr_bh"].round(4)
    return out


def run_logrank_binary_batch(df, binary_vars, time_col=TIME_COL, event_col=EVENT_COL):
    """Run log-rank test for each binary variable (0 vs 1) against time-to-event outcome.
    Returns a DataFrame with test statistic, p-value, and BH-corrected q-value.
    Mirrors run_chi2_batch in structure — added to fix NameError (function was called but never defined).
    """
    from lifelines.statistics import logrank_test

    rows = []
    for v in binary_vars:
        if v not in df.columns:
            continue
        s = pd.to_numeric(df[v], errors="coerce")
        mask0 = s == 0
        mask1 = s == 1
        n0, n1 = int(mask0.sum()), int(mask1.sum())
        if n0 < 2 or n1 < 2:
            rows.append({"Variable": v, "n_0": n0, "n_1": n1,
                         "test_stat": np.nan, "p_logrank": np.nan, "note": "insufficient groups"})
            continue
        g0 = df.loc[mask0]
        g1 = df.loc[mask1]
        try:
            res = logrank_test(
                g0[time_col], g1[time_col],
                event_observed_A=pd.to_numeric(g0[event_col], errors="coerce").fillna(0),
                event_observed_B=pd.to_numeric(g1[event_col], errors="coerce").fillna(0),
            )
            rows.append({
                "Variable":   v,
                "n_0":        n0,
                "n_1":        n1,
                "events_0":   int(pd.to_numeric(g0[event_col], errors="coerce").fillna(0).sum()),
                "events_1":   int(pd.to_numeric(g1[event_col], errors="coerce").fillna(0).sum()),
                "test_stat":  round(float(res.test_statistic), 4),
                "p_logrank":  float(res.p_value),
                "note":       "",
            })
        except Exception as e:
            rows.append({"Variable": v, "n_0": n0, "n_1": n1,
                         "test_stat": np.nan, "p_logrank": np.nan, "note": str(e)})

    out = pd.DataFrame(rows)
    if not out.empty and "p_logrank" in out.columns:
        out["q_fdr_bh"] = fdr_bh(out["p_logrank"].values).round(4)
        out["p_logrank"] = out["p_logrank"].round(4)
    return out


### 2b) Chi-square/Fisher **and** Log-rank for all Cox-aligned covariates (overall + within each cohort)

This section runs:
- **Chi-square / Fisher** for each covariate vs `event` (proportion-based; ignores follow-up time)
- **Log-rank** for each covariate (0 vs 1) using `duration_days` + `event` (time-to-event)

Both are reported **overall** and **within each Cohort** (treatment type).


In [ ]:
def chi2_or_fisher_2way(df, row, col, min_expected=5, dropna=True):
    d = df.copy()
    if dropna:
        d = d.dropna(subset=[row, col])

    ct = pd.crosstab(d[row], d[col], dropna=False)

    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return ct, None

    # chi-square + expected
    chi2, p_chi2, dof, exp = stats.chi2_contingency(ct)
    exp_min = float(np.min(exp))

    # ---- OR calculation for 2x2 (ALWAYS when possible) ----
    odds_ratio = np.nan
    if ct.shape == (2, 2):
        # Ensure consistent ordering (row 0/1, col 0/1) if present
        ct2 = ct.copy()
        # If your binary coding is 0/1, this helps keep alignment
        if set(ct2.index).issuperset({0,1}):
            ct2 = ct2.reindex(index=[0,1])
        if set(ct2.columns).issuperset({0,1}):
            ct2 = ct2.reindex(columns=[0,1])

        a = ct2.iloc[1, 1]  # row=1, col=1
        b = ct2.iloc[1, 0]  # row=1, col=0
        c0 = ct2.iloc[0, 1] # row=0, col=1
        d0 = ct2.iloc[0, 0] # row=0, col=0

        # Haldane-Anscombe correction if any zeros
        if min(a,b,c0,d0) == 0:
            a += 0.5; b += 0.5; c0 += 0.5; d0 += 0.5

        odds_ratio = float((a*d0) / (b*c0))

    # Fisher when sparse (still report OR from fisher_exact too)
    if ct.shape == (2, 2) and exp_min < min_expected:
        oddsratio_f, p_f = stats.fisher_exact(ct.values)
        return ct, {
            "row": row, "col": col,
            "test": "Fisher exact (2x2)",
            "p_value": float(p_f),
            "p_fisher": float(p_f),
            "odds_ratio": float(oddsratio_f),
            "p_chi2": float(p_chi2),
            "chi2": float(chi2),
            "df": int(dof),
            "min_expected": exp_min,
            "note": f"Fisher used (min expected {exp_min:.2f} < {min_expected})"
        }

    # Chi-square branch (now includes OR if 2x2)
    return ct, {
        "row": row, "col": col,
        "test": "Chi-square",
        "p_value": float(p_chi2),
        "p_chi2": float(p_chi2),
        "chi2": float(chi2),
        "df": int(dof),
        "p_fisher": np.nan,
        "odds_ratio": odds_ratio,     # <-- now filled when 2x2
        "min_expected": exp_min,
        "note": "" if exp_min >= min_expected else f"Warning: min expected {exp_min:.2f} < {min_expected}"
    }



# Decide which covariates to test: Cox-aligned binary/dummy variables
candidate_vars = []
if "COX_VARS" in globals():
    candidate_vars = [v for v in COX_VARS if v in episodes.columns]
elif "BIN_VARS" in globals():
    candidate_vars = [v for v in BIN_VARS if v in episodes.columns]
else:
    # fallback: detect 0/1 columns
    candidate_vars = []
    for c in episodes.columns:
        s = pd.to_numeric(episodes[c], errors="coerce")
        u = set(s.dropna().unique().tolist())
        if u.issubset({0, 1}) and len(u) >= 1:
            candidate_vars.append(c)

binary_vars = []
for v in candidate_vars:
    s = pd.to_numeric(episodes[v], errors="coerce")
    u = set(s.dropna().unique().tolist())
    if u.issubset({0, 1}) and len(u) >= 1:
        binary_vars.append(v)

print(f"Chi-square/Log-rank batch will run on {len(binary_vars)} Cox-aligned binary covariates.")

# --- Overall ---
print("\n" + "="*100)
print("OVERALL (all cohorts): Chi-square/Fisher (proportion-based; ignores follow-up time)")
chi_overall = run_chi2_batch(episodes, group_col=None, categorical_vars=binary_vars, event_col=EVENT_COL)
display(chi_overall)

print("\nOVERALL: Log-rank")
lr_overall = run_logrank_binary_batch(episodes, binary_vars)
display(lr_overall)


if "Cohort" in episodes.columns:
    for cohort, sub in episodes.groupby("Cohort", dropna=False):
        print("\n" + "="*100)
        print(f"COHORT: {cohort} | Episodes={len(sub)} | Failures={int(pd.to_numeric(sub[EVENT_COL], errors='coerce').fillna(0).sum())}")

        print("\nChi-square/Fisher within cohort:")
        chi_tbl = run_chi2_batch(sub, group_col=None, categorical_vars=binary_vars, event_col=EVENT_COL)
        chi_tbl.insert(0, "Cohort", cohort)
        display(chi_tbl)

        print("\nLog-rank within cohort (0 vs 1):")
        lr_tbl = run_logrank_binary_batch(sub, binary_vars)   # <-- fixed
        lr_tbl.insert(0, "Cohort", cohort)
        display(lr_tbl)
else:
    print("No 'Cohort' column found; skipping within-cohort batches.")





## OVERALL (all cohorts): Chi-square analysis
Statistically significant univariate associations with the study outcome (proportion-based, ignoring follow-up time).

### Has_Sealing_in_window
Lower occurrence of the study outcome (OR ≈ 0.65, p < 1e-40).

### Has_Crown_in_window
Reduced occurrence of the study outcome (OR ≈ 0.77, p < 1e-14).

### Diabetes
Higher occurrence of the study outcome (OR ≈ 1.55, p < 1e-15).

### Cancer
Increased occurrence of the study outcome (OR ≈ 1.43, p ≈ 0.0002).

---

## OVERALL (all cohorts): Log-rank analysis
Statistically significant differences in time to the study outcome across all cohorts.

### Has_Sealing_in_window
Longer event-free time compared to patients without sealing (p < 0.0001).

### Has_Crown_in_window
Significant difference in time to the study outcome between groups (p < 0.0001).

### Diabetes
Earlier occurrence of the study outcome among patients with diabetes (p < 0.0001).

### Cancer
Earlier occurrence of the study outcome among patients with cancer (p ≈ 0.0002).


## COHORT: Apicoectomy | Episodes=310 | Failures=10

### Chi-square/Fisher within cohort
No covariates were statistically significant after FDR correction (all q_value_fdr_bh ≥ 0.70). Most comparisons required Fisher’s exact test due to sparse counts (minimum expected cell counts < 5), limiting power.

### Log-rank within cohort
No covariates were statistically significant after FDR correction (all q_value_fdr_bh ≥ 0.46). Given the low number of failures (n=10), the cohort-level time-to-event comparisons are underpowered and results should be interpreted cautiously.


## COHORT: Root canal retreatment | Episodes=25015 | Failures=715

### Chi-square/Fisher within cohort
Has_Sealing_in_window and Has_Crown_in_window were significantly associated with a lower occurrence of the study outcome (OR ≈ 0.73, q=0.0010; OR ≈ 0.80, q=0.0140, respectively). Diabetes was significantly associated with a higher occurrence of the study outcome (OR ≈ 1.59, q=0.0016). No other covariates were statistically significant after FDR correction.

### Log-rank within cohort
Time to the study outcome differed significantly by Has_Sealing_in_window and Has_Crown_in_window (q=0.0007 and q=0.0100, respectively), consistent with longer event-free time in the treated groups. Diabetes was also associated with significantly earlier occurrence of the study outcome (q=0.0012). No other covariates were statistically significant after FDR correction.


## COHORT: Root canal treatment | Episodes=119762 | Failures=3490

### Chi-square/Fisher within cohort
Has_Sealing_in_window and Has_Crown_in_window were significantly associated with a lower occurrence of the study outcome (OR ≈ 0.64, q<0.0001; OR ≈ 0.76, q<0.0001, respectively). Cancer and Diabetes were significantly associated with a higher occurrence of the study outcome (OR ≈ 1.48, q=0.0004; OR ≈ 1.54, q<0.0001, respectively). No other covariates were statistically significant after FDR correction.

### Log-rank within cohort
Time to the study outcome differed significantly by Has_Sealing_in_window and Has_Crown_in_window (both q<0.0001), consistent with longer event-free time in the exposed groups. Cancer and Diabetes were associated with significantly earlier occurrence of the study outcome (q=0.0003 and q<0.0001, respectively). No other covariates were statistically significant after FDR correction.


## 3) Kaplan–Meier survival curves


In [ ]:
def km_plot(df, group_col, title, time_col="duration_days", event_col="event", order=None, figsize=(8,5)):
    d = df.copy()
    d[time_col] = pd.to_numeric(d[time_col], errors="coerce") / 365.25  # המרה לשנים
    d[event_col] = pd.to_numeric(d[event_col], errors="coerce").fillna(0).astype(int)
    d = d.dropna(subset=[time_col, group_col])
    d = d[d[time_col] >= 0]
    if d.empty:
        print("No data to plot.")
        return

    kmf = KaplanMeierFitter()
    plt.figure(figsize=figsize)

    groups = order if order is not None else list(pd.Series(d[group_col].unique()))
    for grp in groups:
        g = d[d[group_col] == grp]
        if g.empty:
            continue
        kmf.fit(g[time_col], event_observed=g[event_col], label=f"{grp} (n={len(g)})")
        kmf.plot(ci_show=False)

    plt.title(title)
    plt.xlabel("Years")  # עדכון תווית
    plt.ylabel("Survival probability")
    plt.grid(True, alpha=0.25)
    plt.show()
resto_order = ["Neither","Sealing only","Sealing + Crown"]

# Overall by cohort
km_plot(episodes, "Cohort", title="Survival by treatment type – Endodontic therapy", figsize=(8,5))

# Overall by coronal restoration group
km_plot(episodes, "Coronal_Restoration_Group", title="Survival by coronal restoration – Endodontic therapy",
        order=resto_order, figsize=(8,5))

# Within each cohort: coronal restoration groups
for cohort, sub in episodes.groupby("Cohort"):
    if sub["Coronal_Restoration_Group"].nunique() < 2:
        continue
    km_plot(
        sub, "Coronal_Restoration_Group",
        title=f"Survival by coronal restoration – {cohort}",
        order=resto_order, figsize=(7,4)
    )


## Reviewer comment #81 — Log-rank table with group counts and survival rates

For each restoration group: n episodes, n failures, survival %, plus overall and pairwise log-rank p-values with BH correction.

In [ ]:
# ---- Reviewer comment #81: Full log-rank table (Primary RCT cohort) ----

from lifelines.statistics import multivariate_logrank_test, logrank_test
from itertools import combinations

rct = episodes[episodes['Cohort'] == 'Root canal treatment'].copy()
grp_col = 'Coronal_Restoration_Group'

# ── Per-group summary ─────────────────────────────────────────────────
grp_rows = []
for grp, sub in rct.groupby(grp_col):
    n = len(sub)
    fail = int(sub['event'].sum())
    grp_rows.append({
        'Group': grp,
        'Episodes (n)': n,
        'Failures (n)': fail,
        'Survival (%)': f"{100*(1 - fail/n):.2f}",
    })
grp_summary = pd.DataFrame(grp_rows)

# ── Overall log-rank ──────────────────────────────────────────────────
lr_all = multivariate_logrank_test(rct['duration_days'], rct[grp_col], rct['event'])
print(f"Log-rank test (overall, {grp_col}):")
print(f"  chi2 = {lr_all.test_statistic:.3f},  df = {lr_all.degrees_of_freedom},  "
      f"p = {lr_all.p_value:.4e}")

display(grp_summary)

# ── Pairwise log-rank + BH correction ────────────────────────────────
groups = list(rct[grp_col].dropna().unique())
pair_rows = []
for g1, g2 in combinations(groups, 2):
    s1 = rct[rct[grp_col] == g1]
    s2 = rct[rct[grp_col] == g2]
    res = logrank_test(
        s1['duration_days'], s2['duration_days'],
        event_observed_A=s1['event'], event_observed_B=s2['event']
    )
    pair_rows.append({'Group A': g1, 'Group B': g2, 'p (log-rank)': round(res.p_value, 6)})

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df['q (BH)'] = fdr_bh(pairwise_df['p (log-rank)'].tolist()).round(6)
print('\nPairwise log-rank comparisons (with BH / FDR correction):')
display(pairwise_df)


## 4) Cox proportional hazards models


In [ ]:
from lifelines import CoxPHFitter
import numpy as np
import pandas as pd

def _cox_hr_table(cph: CoxPHFitter) -> pd.DataFrame:
    """Clean Cox output table with HR and 95% CI."""
    s = cph.summary.copy()
    required = ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]
    missing = [c for c in required if c not in s.columns]
    if missing:
        raise KeyError(f"Cox summary missing columns: {missing}. Available: {list(s.columns)}")

    out = pd.DataFrame({
        "Variable": s.index.astype(str),
        "HR": s["exp(coef)"].astype(float),
        "95% CI (lower)": s["exp(coef) lower 95%"].astype(float),
        "95% CI (upper)": s["exp(coef) upper 95%"].astype(float),
        "p_cox": s["p"].astype(float),
    })
    out["HR"] = out["HR"].round(3)
    out["95% CI (lower)"] = out["95% CI (lower)"].round(3)
    out["95% CI (upper)"] = out["95% CI (upper)"].round(3)
    out["p_cox"] = out["p_cox"].round(4)
    return out.sort_values("p_cox")


def build_cox_matrix(df, include_cols, time_col=TIME_COL, event_col=EVENT_COL,
                     min_positive=20,
                     categorical_cols=("Cohort","Coronal_Restoration_Group","AgeGroup"),
                     ref_cols=(
                         "Cohort_Root canal treatment",
                         "Coronal_Restoration_Group_Neither",
                         "AgeGroup_40–60",
                         "AgeGroup_40-60",
                     )):
    d = df.copy()
    d[time_col] = pd.to_numeric(d.get(time_col), errors="coerce")
    d[event_col] = pd.to_numeric(d.get(event_col), errors="coerce").fillna(0).astype(int)
    d = d.dropna(subset=[time_col, event_col])
    d = d[d[time_col] >= 0]

    if d[event_col].sum() == 0:
        return None, "No events -> Cox model not estimable."

    keep=[]
    for c in include_cols:
        if c not in d.columns:
            continue
        if str(d[c].dtype) == "category" or d[c].dtype == "object":
            keep.append(c); continue
        if d[c].nunique(dropna=True) <= 1:
            continue
        if set(pd.to_numeric(d[c], errors='coerce').dropna().unique()).issubset({0,1}) and (pd.to_numeric(d[c], errors='coerce')==1).sum() < min_positive:
            continue
        keep.append(c)

    X = d[[time_col, event_col] + keep].copy()

    for cat in categorical_cols:
        if cat in X.columns:
            X[cat] = X[cat].astype("category")

    X = pd.get_dummies(
        X,
        columns=[c for c in categorical_cols if c in X.columns],
        drop_first=False
    )
    X = X.drop(columns=[c for c in ref_cols if c in X.columns], errors="ignore")

    const_cols = [c for c in X.columns if c not in [time_col, event_col] and X[c].nunique(dropna=True) <= 1]
    X = X.drop(columns=const_cols, errors="ignore")

    return X, None


def fit_cox(df, include_cols, penalizer=0.1, min_positive=20,
            cluster_col=ID_COL, robust=True, check_ph=True):
    """Fit Cox with optional cluster-robust SE and PH check.

    - cluster_col: Patient_ID to account for multiple episodes per patient.
    - robust: robust sandwich SE (recommended when clustering).
    - check_ph: run lifelines PH assumption diagnostic (prints output).
    """
    X, err = build_cox_matrix(df, include_cols, min_positive=min_positive)
    if err:
        return None, None, err
    if X is None or X.empty:
        return None, None, "Empty design matrix."

    # If clustering requested, ensure the column exists in original df and aligns with X
    cph = CoxPHFitter(penalizer=penalizer)

    fit_kwargs = dict(duration_col=TIME_COL, event_col=EVENT_COL)
    if cluster_col in df.columns:
        # align cluster series to X index (X derived from df with drops)
        cluster_series = df.loc[X.index, cluster_col]
        X_fit = X.copy()
        X_fit[cluster_col] = cluster_series.values
        cph.fit(X_fit, **fit_kwargs, robust=robust, cluster_col=cluster_col)
    else:
        cph.fit(X, **fit_kwargs)

    tbl = _cox_hr_table(cph)

    # PH check (diagnostics)
    if check_ph:
        try:
            print("PH assumption check (lifelines):")
            cph.check_assumptions(X, p_value_threshold=0.05, show_plots=False)
        except Exception as e:
            print("PH check failed (non-fatal):", e)

    return cph, tbl, None


base_covars = ["Cohort", "Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols

print("Overall Cox model (all cohorts):")
cph, hr, err = fit_cox(episodes, base_covars, penalizer=0.1, min_positive=20)
if err:
    print(err)
else:
    display(hr)

print("\nCox models within each cohort:")
for cohort, sub in episodes.groupby("Cohort"):
    print("\n" + "=" * 80)
    print(f"{cohort} | Episodes={len(sub)} | Failures={int(sub['event'].sum())}")
    covars = ["Coronal_Restoration_Group", "AgeGroup", "Male"] + systemic_cols
    cph, hr, err = fit_cox(sub, covars)#, penalizer=0.1, min_positive=20)
    if err:
        print(err)
        continue
    display(hr)

## Reviewer comment #84 — Forest plot for Cox model (Primary RCT cohort)

Generates a publication-ready Forest plot from the multivariable Cox HR table for the primary root canal treatment cohort.

In [ ]:
# # ---- Reviewer comment #84: Forest plot — Cox PH, Primary RCT cohort ----

# import matplotlib.pyplot as plt

# # Re-fit Cox for Primary RCT cohort (standalone — does not affect other cells)
# rct = episodes[episodes['Cohort'] == 'Root canal treatment'].copy()
# covars_rct = ['Coronal_Restoration_Group', 'AgeGroup', 'Male'] + systemic_cols
# cph_rct, hr_rct, err_rct = fit_cox(rct, covars_rct, penalizer=0.1, min_positive=20)

# if err_rct:
#     print('Cox model error:', err_rct)
# else:
#     df_fp = hr_rct.copy().reset_index(drop=True)
#     df_fp = df_fp.sort_values('HR', ascending=True).reset_index(drop=True)

#     fig, ax = plt.subplots(figsize=(9, max(5, len(df_fp) * 0.48)))

#     for i, row in df_fp.iterrows():
#         color = 'steelblue' if row['HR'] <= 1 else 'firebrick'
#         # CI line
#         ax.hlines(i, row['95% CI (lower)'], row['95% CI (upper)'],
#                   color=color, linewidth=1.8, zorder=2)
#         # Point estimate
#         ax.plot(row['HR'], i, 'o', color=color, markersize=7,
#                 markeredgecolor='white', markeredgewidth=0.6, zorder=3)

#     # Reference line at HR = 1
#     ax.axvline(x=1.0, color='black', linestyle='--', linewidth=0.9, zorder=1)

#     # y-axis labels
#     ax.set_yticks(range(len(df_fp)))
#     ax.set_yticklabels(df_fp['Variable'].tolist(), fontsize=9)

#     # HR + significance annotation on right
#     x_max = df_fp['95% CI (upper)'].max()
#     for i, row in df_fp.iterrows():
#         sig = '**' if row['p_cox'] < 0.001 else ('*' if row['p_cox'] < 0.05 else '')
#         ax.text(x_max * 1.02, i,
#                 f"{row['HR']:.3f} ({row['95% CI (lower)']:.3f}\u2013{row['95% CI (upper)']:.3f}){sig}",
#                 va='center', fontsize=8, color='black')

#     ax.set_xlabel('Hazard Ratio (95% CI)', fontsize=10)
#     ax.set_title(
#         'Forest Plot \u2014 Multivariable Cox PH Model\n'
#         'Primary Root Canal Treatment Cohort (n = 119,762 episodes)',
#         fontsize=11
#     )
#     ax.xaxis.grid(True, linestyle=':', alpha=0.5)
#     ax.set_xlim(left=max(0.5, df_fp['95% CI (lower)'].min() * 0.88),
#                 right=x_max * 1.35)

#     plt.tight_layout()
#     plt.savefig('forest_plot_primary_rct.png', dpi=150, bbox_inches='tight')
#     plt.show()
#     print('Forest plot saved: forest_plot_primary_rct.png')
#     print('\nHR table used for plot:')
#     display(df_fp[['Variable', 'HR', '95% CI (lower)', '95% CI (upper)', 'p_cox']])


import matplotlib.pyplot as plt

# Re-fit Cox for Primary RCT cohort (standalone — does not affect other cells)
rct = episodes[episodes['Cohort'] == 'Root canal treatment'].copy()
covars_rct = ['Coronal_Restoration_Group', 'AgeGroup', 'Male'] + systemic_cols
cph_rct, hr_rct, err_rct = fit_cox(rct, covars_rct, penalizer=0.1, min_positive=20)


In [ ]:

if err_rct:
    print('Cox model error:', err_rct)
else:
    df_fp = hr_rct.copy().reset_index(drop=True)
    df_fp = df_fp.sort_values('HR', ascending=True).reset_index(drop=True)

    # ---- Clean scientific labels ----
    label_map = {
        'AgeGroup_≥60': 'Age ≥60 years',
        'AgeGroup_<40': 'Age <40 years',
        'Male': 'Male sex',
        'Diabetes': 'Diabetes mellitus',
        'Cancer': 'Malignancy',
        'Smoking': 'Smoking',
        'Hypertension': 'Hypertension',
        'Biphos_use': 'Bisphosphonate use',
        'Coronal_Restoration_Group_Sealing only': 'Coronal sealing only',
        'Coronal_Restoration_Group_Sealing + Crown': 'Coronal sealing + full-coverage crown'
    }

    df_fp['Variable_clean'] = df_fp['Variable'].map(label_map).fillna(df_fp['Variable'])

    fig, ax = plt.subplots(figsize=(9, max(5, len(df_fp) * 0.48)))

    for i, row in df_fp.iterrows():
        color = 'steelblue' if row['HR'] <= 1 else 'firebrick'

        # CI line
        ax.hlines(i, row['95% CI (lower)'], row['95% CI (upper)'],
                  color=color, linewidth=1.8, zorder=2)

        # Point estimate
        ax.plot(row['HR'], i, 'o', color=color, markersize=7,
                markeredgecolor='white', markeredgewidth=0.6, zorder=3)

    # Reference line at HR = 1
    ax.axvline(x=1.0, color='black', linestyle='--', linewidth=0.9, zorder=1)

    # y-axis labels
    ax.set_yticks(range(len(df_fp)))
    ax.set_yticklabels(df_fp['Variable_clean'].tolist(), fontsize=9)

    # HR + significance annotation on right
    x_max = df_fp['95% CI (upper)'].max()
    for i, row in df_fp.iterrows():
        sig = '**' if row['p_cox'] < 0.001 else ('*' if row['p_cox'] < 0.05 else '')
        ax.text(x_max * 1.02, i,
                f"{row['HR']:.3f} ({row['95% CI (lower)']:.3f}\u2013{row['95% CI (upper)']:.3f}){sig}",
                va='center', fontsize=8, color='black')

    # Axis + title (scientific)
    ax.set_xlabel('Hazard Ratio (HR) with 95% Confidence Intervals', fontsize=10)
    ax.set_title(
        'Adjusted Hazard Ratios for Tooth Extraction Following Primary Root Canal Treatment\n'
        'Multivariable Cox Proportional Hazards Model (n = 119,762 Episodes)',
        fontsize=11
    )

    # Grid + limits
    ax.xaxis.grid(True, linestyle=':', alpha=0.5)
    ax.set_xlim(left=max(0.5, df_fp['95% CI (lower)'].min() * 0.88),
                right=x_max * 1.35)

    # Reference note (important for reviewers)
    fig.text(
        0.5, -0.02,
        'Reference categories: No coronal restoration; Age 40–60 years; Female',
        ha='center', fontsize=8
    )

    plt.tight_layout()
    plt.savefig('forest_plot_primary_rct.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Forest plot saved: forest_plot_primary_rct.png')
    print('\nHR table used for plot:')
    display(df_fp[['Variable_clean', 'HR', '95% CI (lower)', '95% CI (upper)', 'p_cox']])

## OVERALL (all cohorts): Cox proportional hazards model

The multivariable Cox model (adjusted for covariates listed) identified several independent associations with time to the study outcome. Diabetes was associated with a higher hazard of the outcome (HR=1.097, 95% CI 1.057–1.138, p<0.001). Compared with the reference restoration category, both coronal restoration groups were associated with a lower hazard (Sealing + Crown: HR=0.938, 95% CI 0.925–0.951, p<0.001; Sealing only: HR=0.959, 95% CI 0.945–0.973, p<0.001). Age showed a strong gradient: Age <40 was associated with lower hazard (HR=0.878, 95% CI 0.866–0.890, p<0.001), whereas Age ≥60 was associated with higher hazard (HR=1.117, 95% CI 1.091–1.144, p<0.001). Smoking (HR=1.024, 95% CI 1.003–1.047, p=0.026) and cancer (HR=1.074, 95% CI 1.008–1.143, p=0.027) were associated with modestly higher hazards. Sex, hypertension, bisphosphonate use, and cohort indicators were not statistically significant in this model.

The proportional hazards (PH) diagnostic flagged non-proportionality related to Patient_ID (reported as a non-fatal check failure), suggesting potential within-patient correlation and/or time-varying effects; results should therefore be interpreted with caution and may warrant a clustered/robust variance or frailty specification.

---

## COHORT: Apicoectomy | Episodes=310 | Failures=10 — Cox model

Within Apicoectomy, Age ≥60 was associated with a lower hazard of the outcome (HR=0.746, 95% CI 0.630–0.883, p=0.0007). No other covariates were statistically significant. Given the small number of failures (n=10), estimates are likely underpowered and should be interpreted cautiously. The PH diagnostic also flagged Patient_ID.

---

## COHORT: Root canal retreatment | Episodes=25015 | Failures=715 — Cox model

In the retreatment cohort, age was strongly associated with outcome risk: Age <40 was protective (HR=0.880, 95% CI 0.853–0.907, p<0.001), while Age ≥60 increased hazard (HR=1.103, 95% CI 1.046–1.164, p=0.0003). Coronal restoration (Sealing + Crown) was associated with reduced hazard (HR=0.948, 95% CI 0.918–0.979, p=0.0011). Diabetes was associated with higher hazard (HR=1.106, 95% CI 1.020–1.201, p=0.015). Other covariates were not statistically significant. The PH diagnostic flagged Patient_ID.

---

## COHORT: Root canal treatment | Episodes=119762 | Failures=3490 — Cox model

In the primary root canal treatment cohort, Diabetes was associated with increased hazard (HR=1.094, 95% CI 1.054–1.137, p<0.001). Both coronal restoration groups were associated with reduced hazard (Sealing + Crown: HR=0.935, 95% CI 0.921–0.949, p<0.001; Sealing only: HR=0.956, 95% CI 0.941–0.972, p<0.001). Age <40 was protective (HR=0.877, 95% CI 0.865–0.890, p<0.001), whereas Age ≥60 increased hazard (HR=1.121, 95% CI 1.093–1.149, p<0.001). Cancer (HR=1.085, 95% CI 1.015–1.161, p=0.0167) and smoking (HR=1.027, 95% CI 1.004–1.051, p=0.021) were associated with modestly higher hazards. Sex, bisphosphonate use, and hypertension were not statistically significant. The PH diagnostic flagged Patient_ID.
